[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# autocommit and isolation_level


## What you will be able to do

Say exactly when sqlite3 begins and ends a transaction under each of its three settings: legacy
control, which is still the default, `autocommit=False` and `autocommit=True`. Watch the `BEGIN`,
`COMMIT` and `ROLLBACK` statements sqlite3 adds, with a trace callback. Predict whether a `rollback`
undoes a `CREATE TABLE`, choose `autocommit=False` for new code, and recognize the settings that
sqlite3 accepts and then ignores.


## The idea

### The problem

The **Transactions** notebook took one rule on trust: sqlite3 begins a transaction before an
`INSERT`, `UPDATE`, `DELETE` or `REPLACE`. The rule is exact, and what it leaves out matters. A
`CREATE TABLE` is none of those four, so a loader that creates a staging table, fills it, finds a
bad reading and rolls back keeps the table, empty, and the loader's next run fails because the table
already exists. Yet the same rollback does undo the same `CREATE TABLE` when an insert happened to
run first, because the table then joined the insert's transaction.

That default is what Python's documentation calls legacy transaction control: the behavior sqlite3
had before Python 3.12, which does not follow PEP 249, the specification that Python's database
drivers share. Python 3.12 added an `autocommit` setting to replace it. With `autocommit=False` a
transaction is always open, so everything runs inside one, `CREATE TABLE` included, and the
documentation recommends it. With `autocommit=True` sqlite3 begins no transactions at all. And the
older setting, `isolation_level`, still sits beside both, meaning something under legacy control and
nothing under either of the others.

### What transaction control is

> **Transaction control** is the set of rules by which sqlite3 begins and ends transactions for you.
> The **`autocommit`** setting, a parameter of `connect` and an attribute of the connection since
> Python 3.12, chooses among three. **`sqlite3.LEGACY_TRANSACTION_CONTROL`**, the default, sends a
> `BEGIN` before an `INSERT`, `UPDATE`, `DELETE` or `REPLACE` when no transaction is open, of the
> kind that **`isolation_level`** names: `'DEFERRED'`, which `''` also means, `'IMMEDIATE'` or
> `'EXCLUSIVE'`, or no `BEGIN` at all for `None`. **`False`** keeps a transaction open at all times,
> so `commit` and `rollback` each end one and begin the next. **`True`** sends no `BEGIN` of its own,
> so every statement is saved as it runs unless the SQL begins a transaction itself.

### Why it works that way

- **Legacy control begins a transaction only for changes to rows.** A `SELECT`, a `CREATE TABLE` or
  a `PRAGMA` run with no transaction open runs as a transaction of its own, so a `CREATE TABLE` is
  saved as it runs. Once a change to rows has begun a transaction, the same statements run inside it:
  Python's documentation notes that sqlite3 used to commit an open transaction before such a
  statement, and no longer does.
- **`autocommit=False` is PEP 249's model.** The connection begins a transaction when it connects,
  and again after every `commit` and `rollback`, so all work, schema changes included, waits for a
  commit. Python's documentation calls `False` the recommended value, and says the default will
  change to it in a future release.
- **`autocommit=True` leaves transactions to SQL.** Every statement is saved as it runs, `commit()`
  and `rollback()` do nothing, even inside a `BEGIN` that the code sent itself, and a transaction
  lasts from a `BEGIN` to a `COMMIT` or `ROLLBACK` written in SQL.
- **Some statements cannot run inside a transaction.** `VACUUM` refuses, and some `PRAGMA` settings
  are silently ignored there, which matters under `autocommit=False`, where a transaction is always
  open. Setting `conn.autocommit` to `True` commits and leaves transactions behind, and setting it
  back to `False` begins a new one.
- **`isolation_level` only chooses the kind of `BEGIN`.** It has an effect under legacy control, and
  under `autocommit=False` or `True` it is accepted, reported back, and ignored.
- **A trace callback shows what sqlite3 sends.** `conn.set_trace_callback` calls a function with
  every statement SQLite runs, including the `BEGIN`, `COMMIT` and `ROLLBACK` that sqlite3 adds.

### Where this shows up

PEP 249 is why every Python database driver has `commit` and `rollback`, and the drivers still
disagree about when a transaction begins. psycopg, in the **asyncpg and psycopg3, Deep Dive** guide,
begins one with the first statement, as `autocommit=False` does, and has an `autocommit` attribute of
its own for the commands that cannot run inside a transaction. asyncpg, in the same guide, runs every
statement in autocommit until the code opens a transaction. In this guide, the **Constraints**
notebook shows a `PRAGMA` that an open transaction silently ignores, the **Changing a Schema**
notebook needs several schema changes to succeed or fail together, and the **Concurrency and WAL**
notebook shows what a `BEGIN IMMEDIATE` is for.

### What this notebook covers

- The three settings, and what `autocommit`, `isolation_level` and `in_transaction` report
- A trace of the statements sqlite3 adds
- Legacy control: the `CREATE TABLE` a rollback cannot undo, and the one it can
- `autocommit=False`: a transaction that is always open, and a moment outside it for `VACUUM`
- `autocommit=True`: transactions written in SQL
- `isolation_level`, which chooses the kind of `BEGIN`
- `executescript` under each setting
- When to use `autocommit=False`, legacy control or `autocommit=True`
- A load through a staging table, in one transaction
- Five errors: an isolation level sqlite3 does not accept, `None` for `autocommit`, a script with its
  own `BEGIN`, a staging table a rollback left behind, and `isolation_level` beside `autocommit=True`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

for setting in [sqlite3.LEGACY_TRANSACTION_CONTROL, False]:
    conn = sqlite3.connect(":memory:", autocommit=setting)
    conn.execute("CREATE TABLE staging (celsius REAL)")
    conn.execute("INSERT INTO staging VALUES (93.0)")
    conn.rollback()
    tables = conn.execute("SELECT name FROM sqlite_schema").fetchall()
    print(f"autocommit={setting!r:<6} tables after rollback: {tables}")
    conn.close()
```

```
autocommit=-1     tables after rollback: [('staging',)]
autocommit=False  tables after rollback: []
```

The same three statements under two settings. Under the default, legacy control, the `CREATE TABLE`
ran before any transaction was open, so SQLite saved it at once, and the rollback undid only the
insert. Under `autocommit=False` a transaction was already open when the table was created, so the
rollback undid the table along with the insert.


## Setup

Five imports, and the stations' year, built into the two tables the **Tables and Queries** notebook
designed.

- `sqlite3` builds the database, runs every statement and holds the transaction settings
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `Path` names the scratch folder and the database in it
- `shutil` removes the scratch folder at the end


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
    CREATE TABLE audit (note TEXT NOT NULL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()

print("built", DATABASE)


built scratch/stations.db


## Worked examples

### The three settings, and what they report

A connection reports its settings as attributes: `autocommit`, `isolation_level`, and
`in_transaction`, which says whether a transaction is open. Four connections, opened four ways:


In [2]:
for options in [{}, {"autocommit": False}, {"autocommit": True}, {"isolation_level": None}]:
    conn = sqlite3.connect(DATABASE, **options)
    print(f"{str(options):<26} autocommit={conn.autocommit!r:<6} isolation_level={conn.isolation_level!r:<5}"
          f" in_transaction={conn.in_transaction}")
    conn.close()

print("sqlite3.LEGACY_TRANSACTION_CONTROL is", sqlite3.LEGACY_TRANSACTION_CONTROL)


{}                         autocommit=-1     isolation_level=''    in_transaction=False
{'autocommit': False}      autocommit=False  isolation_level=''    in_transaction=True
{'autocommit': True}       autocommit=True   isolation_level=''    in_transaction=False
{'isolation_level': None}  autocommit=-1     isolation_level=None  in_transaction=False
sqlite3.LEGACY_TRANSACTION_CONTROL is -1


A connection that chose nothing reports `autocommit` as -1, the value of
`sqlite3.LEGACY_TRANSACTION_CONTROL`, and `isolation_level` as `''`, which means `'DEFERRED'`. The
connection opened with `autocommit=False` was already in a transaction before it ran a statement.
`isolation_level=None` is the older way to stop legacy control from sending `BEGIN`, and it leaves
`autocommit` at legacy control. `autocommit` arrived in Python 3.12, so an older Python rejects the
keyword with a `TypeError`.

### What sqlite3 sends, with a trace

`set_trace_callback` takes a function and calls it with every statement SQLite runs, with the values
of its parameters written in. `run_and_trace` runs some statements, rolls back, and returns every
statement that was sent, so the `BEGIN` and `ROLLBACK` that sqlite3 adds show up among the ones the
code wrote. `table_exists` asks `sqlite_schema` whether a table is there, and `checker` is a
connection kept only for looking, which never holds a transaction open, so it never stands in the
way of another connection's commit:


In [3]:
def run_and_trace(conn, statements):
    """Run (sql, parameters) pairs in order, roll back, and return every statement SQLite ran."""
    sent = []
    conn.set_trace_callback(sent.append)
    for sql, parameters in statements:
        conn.execute(sql, parameters)
    conn.rollback()
    conn.set_trace_callback(None)
    return [statement.strip() for statement in sent]


def table_exists(conn, name):
    """Whether the database has a table of this name."""
    return conn.execute("SELECT COUNT(*) FROM sqlite_schema WHERE type = 'table' AND name = ?", (name,)).fetchone()[0] == 1


legacy = sqlite3.connect(DATABASE)
checker = sqlite3.connect(DATABASE, autocommit=True)     # looks, and never holds a transaction open
station_ids = dict(legacy.execute("SELECT name, id FROM stations"))

for statement in run_and_trace(legacy, [
    ("SELECT COUNT(*) FROM audit", ()),
    ("INSERT INTO audit VALUES (?)", ("a note",)),
    ("UPDATE audit SET note = ? WHERE note = ?", ("an edited note", "a note")),
]):
    print("sent:", statement)


sent: SELECT COUNT(*) FROM audit
sent: BEGIN
sent: INSERT INTO audit VALUES ('a note')
sent: UPDATE audit SET note = 'an edited note' WHERE note = 'a note'
sent: ROLLBACK


The query went alone. Before the insert, sqlite3 sent `BEGIN` of its own, the update ran inside the
transaction the insert had begun, and `rollback()` sent `ROLLBACK`. The trace shows the parameters
written into the insert, as SQLite ran it.

### Legacy control: the CREATE TABLE a rollback cannot undo

The same two statements, a `CREATE TABLE` and an insert into the new table, run twice on the legacy
connection: first with nothing pending, then after an insert into `audit` has already begun a
transaction:


In [4]:
print("sent:", run_and_trace(legacy, [
    ("CREATE TABLE staging (celsius REAL)", ()),
    ("INSERT INTO staging VALUES (?)", (93.0,)),
]))
print("staging after rollback:", table_exists(legacy, "staging"))
legacy.execute("DROP TABLE staging")

print("sent:", run_and_trace(legacy, [
    ("INSERT INTO audit VALUES (?)", ("written first",)),
    ("CREATE TABLE staging (celsius REAL)", ()),
    ("INSERT INTO staging VALUES (?)", (93.0,)),
]))
print("staging after rollback:", table_exists(legacy, "staging"))


sent: ['CREATE TABLE staging (celsius REAL)', 'BEGIN', 'INSERT INTO staging VALUES (93.0)', 'ROLLBACK']
staging after rollback: True
sent: ['BEGIN', "INSERT INTO audit VALUES ('written first')", 'CREATE TABLE staging (celsius REAL)', 'INSERT INTO staging VALUES (93.0)', 'ROLLBACK']
staging after rollback: False


In the first run, the `CREATE TABLE` went before any `BEGIN`, so SQLite saved it as a transaction of
its own, and the `ROLLBACK` could undo only the insert: the table survived, empty. In the second run,
the insert into `audit` had already brought the `BEGIN`, so the `CREATE TABLE` ran inside that
transaction, and the `ROLLBACK` removed the table too. Whether a rollback undoes a schema change
depends on what happened to run before it, which is the trouble with legacy control.

### autocommit=False: a transaction that is always open

Under `autocommit=False`, sqlite3 begins a transaction as the connection opens and again after every
`commit` and `rollback`, so the same `CREATE TABLE` always waits for a commit:


In [5]:
pep249 = sqlite3.connect(DATABASE, autocommit=False)
print("in a transaction on connecting:", pep249.in_transaction)

print("sent:", run_and_trace(pep249, [
    ("CREATE TABLE staging (celsius REAL)", ()),
    ("INSERT INTO staging VALUES (?)", (93.0,)),
]))
print("staging after rollback:", table_exists(pep249, "staging"))
print("in a transaction after rollback:", pep249.in_transaction)


in a transaction on connecting: True
sent: ['CREATE TABLE staging (celsius REAL)', 'INSERT INTO staging VALUES (93.0)', 'ROLLBACK', 'BEGIN']
staging after rollback: False
in a transaction after rollback: True


No `BEGIN` appeared before the `CREATE TABLE`, since a transaction was already open, and the
`ROLLBACK` removed the table. sqlite3 then sent a `BEGIN` at once, so the connection is in a
transaction again. Under this setting `close()` also rolls back whatever is pending.

A transaction that is always open has one cost: a statement that cannot run inside a transaction has
nowhere to run. `VACUUM`, which rebuilds the database file, is one. Setting `autocommit` to `True`
commits and leaves transactions behind for that one statement, and setting it back to `False` begins
a new transaction:


In [6]:
try:
    pep249.execute("VACUUM")
except sqlite3.OperationalError as error:
    print("VACUUM inside the transaction:", error)

pep249.autocommit = True
print("in a transaction with autocommit=True: ", pep249.in_transaction)
pep249.execute("VACUUM")
pep249.autocommit = False
print("in a transaction with autocommit=False:", pep249.in_transaction)


VACUUM inside the transaction: cannot VACUUM from within a transaction
in a transaction with autocommit=True:  False
in a transaction with autocommit=False: True


### autocommit=True: transactions written in SQL

Under `autocommit=True`, sqlite3 sends nothing of its own. Every statement is saved as it runs, and a
transaction is whatever the SQL makes one. `commit()` and `rollback()` do nothing, even inside a
`BEGIN` the code sent itself, as the trace shows:


In [7]:
manual = sqlite3.connect(DATABASE, autocommit=True)
sent = []
manual.set_trace_callback(sent.append)

manual.execute("INSERT INTO audit VALUES (?)", ("saved as it ran",))
manual.rollback()
manual.execute("BEGIN")
manual.execute("INSERT INTO audit VALUES (?)", ("inside a BEGIN",))
manual.commit()
print("in a transaction after commit():", manual.in_transaction)
manual.execute("ROLLBACK")

manual.set_trace_callback(None)
print("sent:", [statement.strip() for statement in sent])
print("notes kept:", manual.execute("SELECT note FROM audit").fetchall())


in a transaction after commit(): True
sent: ["INSERT INTO audit VALUES ('saved as it ran')", 'BEGIN', "INSERT INTO audit VALUES ('inside a BEGIN')", 'ROLLBACK']
notes kept: [('saved as it ran',)]


The first insert was saved as it ran, so `rollback()` had nothing to undo, and sent nothing. After
the code's own `BEGIN`, `commit()` still sent nothing and left the transaction open, and only the
`ROLLBACK` written in SQL ended it, throwing away the second note. Under this setting, the SQL is
the only thing that begins or ends a transaction.

### isolation_level: the kind of BEGIN

Under legacy control, `isolation_level` decides which `BEGIN` sqlite3 sends. SQLite has three:
`DEFERRED` takes no lock until the transaction first reads or writes, `IMMEDIATE` takes the write
lock at once, and `EXCLUSIVE` goes further and, in SQLite's default journal mode, keeps other
connections from reading too. `None` sends no `BEGIN` at all:


In [8]:
for level in ["", "DEFERRED", "IMMEDIATE", "EXCLUSIVE", None]:
    conn = sqlite3.connect(DATABASE, isolation_level=level)
    print(f"{level!r:<12}", run_and_trace(conn, [("INSERT INTO audit VALUES (?)", ("a note",))]))
    conn.close()

print("notes kept:", checker.execute("SELECT COUNT(*) FROM audit WHERE note = 'a note'").fetchone()[0])


''           ['BEGIN', "INSERT INTO audit VALUES ('a note')", 'ROLLBACK']
'DEFERRED'   ['BEGIN DEFERRED', "INSERT INTO audit VALUES ('a note')", 'ROLLBACK']
'IMMEDIATE'  ['BEGIN IMMEDIATE', "INSERT INTO audit VALUES ('a note')", 'ROLLBACK']
'EXCLUSIVE'  ['BEGIN EXCLUSIVE', "INSERT INTO audit VALUES ('a note')", 'ROLLBACK']
None         ["INSERT INTO audit VALUES ('a note')"]
notes kept: 1


`''` sent a plain `BEGIN`, which SQLite treats as `DEFERRED`, and the other three sent their names.
With `None` the insert ran with no `BEGIN`, so it was saved as it ran, and the rollback that followed
had nothing to undo, which is why one note was kept. The **Concurrency and WAL** notebook shows when
taking the write lock with `IMMEDIATE` avoids an error that waiting cannot.

### executescript under each setting

`executescript` follows the setting too. Under legacy control it commits a pending transaction before
the script, under `autocommit=False` it runs the script inside the open transaction, and under
`autocommit=True` there is nothing pending to commit. Here every setting inserts a note, runs a
one-line script, and rolls back:


In [9]:
for options in [{}, {"autocommit": False}, {"autocommit": True}]:
    conn = sqlite3.connect(DATABASE, **options)
    conn.execute("INSERT INTO audit VALUES (?)", ("before the script",))
    sent = []
    conn.set_trace_callback(sent.append)
    conn.executescript("INSERT INTO audit VALUES ('from the script');")
    conn.set_trace_callback(None)
    conn.rollback()

    kept = conn.execute("SELECT COUNT(*) FROM audit WHERE note IN ('before the script', 'from the script')").fetchone()[0]
    print(f"{str(options):<22} sent {[statement.strip() for statement in sent]}, notes kept after rollback: {kept}")
    conn.execute("DELETE FROM audit WHERE note IN ('before the script', 'from the script')")
    conn.commit()
    conn.close()


{}                     sent ['COMMIT', "INSERT INTO audit VALUES ('from the script');"], notes kept after rollback: 2
{'autocommit': False}  sent ["INSERT INTO audit VALUES ('from the script');"], notes kept after rollback: 0
{'autocommit': True}   sent ["INSERT INTO audit VALUES ('from the script');"], notes kept after rollback: 2


Legacy control sent a `COMMIT` before the script, which is how a later rollback found nothing to
undo, as the **Transactions** notebook showed. Under `autocommit=False` the script ran inside the
open transaction, and the rollback undid both notes. Under `autocommit=True` both notes were saved
as they ran.

### autocommit=False, legacy control, or autocommit=True

Each setting suits different code:

| Write | When | Why |
|---|---|---|
| `autocommit=False` | new code | a transaction is always open, so every change, a `CREATE TABLE` included, waits for a commit, and Python's documentation recommends it |
| legacy control, the default | code written for sqlite3 before Python 3.12 that depends on its rules | changing the setting changes when that code's changes are saved |
| `autocommit=True` | code that writes its own `BEGIN` and `COMMIT`, or a moment for `VACUUM` or a `PRAGMA` that cannot run inside a transaction | sqlite3 adds nothing, so the SQL alone decides |

The default for new code is `autocommit=False`, passed to `connect` explicitly, since Python's own
default is expected to change. `isolation_level` then has no job to do, and is best left alone.

### A load through a staging table, in one transaction

The pieces of this notebook in one job: readings arrive in a batch, and go into a staging table
first, so they can be checked before any of them reaches `readings`. Creating the staging table,
filling it, checking it, moving its rows and dropping it are one transaction under
`autocommit=False`, in a `with` block, so a batch that fails the check leaves nothing behind, the
staging table included:


In [10]:
def load_through_staging(conn, rows):
    """Create a staging table, fill it, check it, move its rows into readings and drop it, as one transaction."""
    with conn:
        conn.execute("CREATE TABLE staging (station_id INTEGER NOT NULL, hour TEXT NOT NULL, celsius REAL) STRICT")
        conn.executemany("INSERT INTO staging VALUES (?, ?, ?)", rows)
        implausible = conn.execute("SELECT COUNT(*) FROM staging WHERE celsius NOT BETWEEN -60 AND 60").fetchone()[0]
        if implausible:
            raise ValueError(f"{implausible} implausible reading in the batch, so none of it is loaded")
        conn.execute("INSERT INTO readings (station_id, hour, celsius) SELECT station_id, hour, celsius FROM staging")
        conn.execute("DROP TABLE staging")
    return len(rows)


def kirkenes_readings(conn, day):
    """How many readings Kirkenes has on a day."""
    return conn.execute("SELECT COUNT(*) FROM readings WHERE station_id = ? AND hour LIKE ?",
                        (station_ids["Kirkenes"], f"{day}%")).fetchone()[0]


def after_the_load(day):
    """Whether a staging table was left behind, and a day's Kirkenes readings, as checker sees them."""
    left_behind = table_exists(checker, "staging")
    return f"staging table left behind: {left_behind} | readings on {day}: {kirkenes_readings(checker, day)}"


batch = [(station_ids["Kirkenes"], f"2025-12-08T{hour:02d}:00", celsius)
         for hour, celsius in enumerate([-9.9, -10.2, 93.0, -10.4])]
try:
    load_through_staging(pep249, batch)
except ValueError as error:
    print("refused:", error)
print(after_the_load("2025-12-08"))

batch[2] = (station_ids["Kirkenes"], "2025-12-08T02:00", -10.3)
print("loaded:", load_through_staging(pep249, batch))
print(after_the_load("2025-12-08"))


refused: 1 implausible reading in the batch, so none of it is loaded
staging table left behind: False | readings on 2025-12-08: 0
loaded: 4
staging table left behind: False | readings on 2025-12-08: 4


The batch with a reading of 93.0 degrees was refused, and the `with` block's rollback took the
staging table away with the rows, since under `autocommit=False` the `CREATE TABLE` was inside the
transaction. The corrected batch went through all five steps and was committed at the end of the
block, and the connection was straight away in a new transaction for whatever came next.

### Where each part came from

| In the load | What it relies on | The section that showed it |
|---|---|---|
| `connect(DATABASE, autocommit=False)` for `pep249` | a transaction that is always open | autocommit=False: a transaction that is always open |
| `CREATE TABLE staging` inside the transaction | a schema change that a rollback undoes | Legacy control: the CREATE TABLE a rollback cannot undo |
| `with conn:` raising `ValueError` | a rollback, and a new transaction after it | autocommit=False: a transaction that is always open |
| `table_exists` after the refusal | `sqlite_schema` naming what is really there | What sqlite3 sends, with a trace |
| no `isolation_level` anywhere | a setting with no effect outside legacy control | isolation_level: the kind of BEGIN |
| `autocommit=False` as the choice | the default for new code | autocommit=False, legacy control, or autocommit=True |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/09-autocommit-and-isolation-level-solutions.ipynb).

**1.** Open a connection with `isolation_level=None`, print its `autocommit` and `isolation_level`,
insert a note into `audit`, and print `in_transaction`.


In [11]:
# your code here


**2.** Put a trace callback that prints every statement on a legacy connection, run a query, an
insert and `commit()`, and name the statements that sqlite3 added.


In [12]:
# your code here


**3.** Under legacy control, create a table, insert a row into it, roll back, and show that the table
is there and empty. Then do the same under `autocommit=False`, and show that the table is gone.


In [13]:
# your code here


**4.** On an `autocommit=True` connection, write `BEGIN`, insert two notes and write `ROLLBACK`, and
show that neither was saved. Then insert a note with no `BEGIN`, call `rollback()`, and show that it
was.


In [14]:
# your code here


**5.** On an `autocommit=False` connection, run `VACUUM` by switching `autocommit` for that one
statement, printing `in_transaction` before, during and after.


In [15]:
# your code here


**6.** On an `autocommit=False` connection, run a script that inserts two notes with `executescript`,
roll back, and show that neither note was saved.


In [16]:
# your code here


## Common errors

### ValueError: isolation_level string must be '', 'DEFERRED', 'IMMEDIATE', or 'EXCLUSIVE'


In [17]:
sqlite3.connect(DATABASE, isolation_level="SERIALIZABLE")


ValueError: isolation_level string must be '', 'DEFERRED', 'IMMEDIATE', or 'EXCLUSIVE'

`SERIALIZABLE` is the name of an isolation level in the SQL standard and in PostgreSQL, where it
chooses how much concurrent transactions may see of each other. SQLite's transactions are already
serializable, and in sqlite3, `isolation_level` names the kind of `BEGIN` to send, not an isolation
level, so it accepts only SQLite's three kinds of `BEGIN`, `''` or `None`. For new code, choose
`autocommit=False` and leave `isolation_level` out:


In [18]:
conn = sqlite3.connect(DATABASE, autocommit=False)
print("autocommit:", conn.autocommit, "| in a transaction:", conn.in_transaction)
conn.close()


autocommit: False | in a transaction: True


### ValueError: autocommit must be True, False, or sqlite3.LEGACY_TRANSACTION_CONTROL


In [19]:
sqlite3.connect(DATABASE, autocommit=None)


ValueError: autocommit must be True, False, or sqlite3.LEGACY_TRANSACTION_CONTROL

`isolation_level=None` switches off sqlite3's `BEGIN`, and it is natural to guess that `autocommit`
takes `None` the same way, but `autocommit` accepts exactly three values, and `None`, `0` or the
string `'False'` from a configuration file are none of them. The setting that sends no `BEGIN` is
`autocommit=True`:


In [20]:
conn = sqlite3.connect(DATABASE, autocommit=True)
print("autocommit:", conn.autocommit, "| in a transaction:", conn.in_transaction)
conn.close()


autocommit: True | in a transaction: False


### sqlite3.OperationalError: cannot start a transaction within a transaction


In [21]:
migration = """
    BEGIN;
    INSERT INTO audit VALUES ('migration started');
    COMMIT;
"""
pep249.executescript(migration)


OperationalError: cannot start a transaction within a transaction

The script opens its own transaction with `BEGIN`, which suits legacy control and `autocommit=True`.
Under `autocommit=False` a transaction is already open, and `executescript` runs the script inside
it, so the script's `BEGIN` fails. Take the transaction statements out of the script and commit from
Python, or run a script that must keep its own `BEGIN` with `autocommit` switched to `True`:


In [22]:
pep249.rollback()
pep249.executescript("INSERT INTO audit VALUES ('migration started');")
pep249.commit()

print(checker.execute("SELECT note FROM audit WHERE note = 'migration started'").fetchall())


[('migration started',)]


### sqlite3.OperationalError: table staging already exists


In [23]:
batch = [(station_ids["Kirkenes"], f"2025-12-09T{hour:02d}:00", celsius) for hour, celsius in enumerate([-11.0, 110.0])]
try:
    load_through_staging(legacy, batch)
except ValueError as error:
    print("the first run refused:", error)

batch[1] = (station_ids["Kirkenes"], "2025-12-09T01:00", -11.2)
load_through_staging(legacy, batch)


the first run refused: 1 implausible reading in the batch, so none of it is loaded


OperationalError: table staging already exists

The same loader, on the legacy connection. Its first run created the staging table before any
`BEGIN`, so SQLite saved the table at once, and the rollback from the `with` block removed only the
rows. The table stayed behind, empty, and the second run's `CREATE TABLE` failed on it, even though
the batch was now fine. Remove the leftover table, then give the connection `autocommit=False`, so
the next rollback takes the table with it:


In [24]:
legacy.execute("DROP TABLE staging")
legacy.autocommit = False

print("loaded:", load_through_staging(legacy, batch))
print(after_the_load("2025-12-09"))


loaded: 2
staging table left behind: False | readings on 2025-12-09: 2


### No error, and a rollback that undid nothing: isolation_level beside autocommit=True


In [25]:
careful_looking = sqlite3.connect(DATABASE, autocommit=True, isolation_level="IMMEDIATE")
print("isolation_level:", careful_looking.isolation_level)

careful_looking.execute("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                        (station_ids["Kirkenes"], "2025-12-10T00:00", 93.0))       # a mistake, to be rolled back
careful_looking.rollback()
print("readings on 10 December after rollback():", kirkenes_readings(checker, "2025-12-10"))


isolation_level: IMMEDIATE
readings on 10 December after rollback(): 1


The connection reported `isolation_level` as `IMMEDIATE`, which suggests that sqlite3 begins
immediate transactions, and it does not: under `autocommit=True`, `isolation_level` is accepted,
reported back and ignored. No `BEGIN` was sent, the mistaken reading was saved as it ran, and
`rollback()` did nothing. Nothing raised at any point. Choose the transaction setting with
`autocommit` alone, and write any `BEGIN IMMEDIATE` in SQL:


In [26]:
careful_looking.execute("DELETE FROM readings WHERE station_id = ? AND hour = ?",
                        (station_ids["Kirkenes"], "2025-12-10T00:00"))
careful_looking.close()

careful = sqlite3.connect(DATABASE, autocommit=False)
careful.execute("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                (station_ids["Kirkenes"], "2025-12-10T00:00", 93.0))
careful.rollback()
print("readings on 10 December after rollback():", kirkenes_readings(checker, "2025-12-10"))

for conn in (careful, manual, pep249, legacy, checker):
    conn.close()


readings on 10 December after rollback(): 0


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
database in it:


In [27]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `autocommit`, since Python 3.12, chooses how sqlite3 handles transactions: legacy control by
  default, `False`, or `True`, and nothing else.
- Legacy control sends `BEGIN` only before `INSERT`, `UPDATE`, `DELETE` and `REPLACE`, so a
  `CREATE TABLE` run with nothing pending is saved at once and survives a rollback.
- `autocommit=False` keeps a transaction open at all times, so a rollback undoes a `CREATE TABLE`
  too. It is the recommended setting, and the expected future default.
- `autocommit=True` sends nothing: statements are saved as they run, `commit()` and `rollback()` do
  nothing, and transactions are `BEGIN` and `COMMIT` written in SQL.
- `isolation_level` chooses the kind of `BEGIN` under legacy control, and is ignored under the other
  two settings, even when it is reported back.
- `executescript` commits a pending transaction first only under legacy control, and a statement
  such as `VACUUM` needs a moment with `autocommit=True` when a transaction is always open.
- `set_trace_callback` shows every statement SQLite runs, sqlite3's own `BEGIN`, `COMMIT` and
  `ROLLBACK` included.


## What is next

The **Constraints** notebook turns from when changes are saved to which changes a table refuses:
`UNIQUE` and foreign keys, the `PRAGMA` that switches foreign keys on, and the upsert that answers a
duplicate.


---

&#8592; **Previous:** [Transactions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/08-transactions.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Constraints](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/10-constraints.ipynb) &#8594;
